In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

# 1. Load Dataset
df = pd.read_csv("Dataset .csv")
df.columns = df.columns.str.strip()

# 2. Handle Missing Values & Combine Features
df['Cuisines'] = df['Cuisines'].fillna('Unknown')
df['Price range'] = df['Price range'].fillna(df['Price range'].median())
df['Aggregate rating'] = df['Aggregate rating'].fillna(0)

df['combined_features'] = df['Cuisines'].astype(str) + " PriceRange_" + df['Price range'].astype(str)

# 3. Vectorization & Normalization
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['combined_features'])

scaler = MinMaxScaler()
normalized_ratings = scaler.fit_transform(df[['Aggregate rating']]).flatten()

# 4. Recommendation Function
def recommend_restaurants(user_cuisine, user_price_range, top_n=5):
    query_str = f"{user_cuisine} PriceRange_{user_price_range}"
    query_vec = tfidf.transform([query_str])
    
    sim_scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
    final_scores = (0.7 * sim_scores) + (0.3 * normalized_ratings)
    
    top_indices = np.argsort(final_scores)[::-1][:top_n]
    
    results = df.iloc[top_indices][['Restaurant Name', 'Cuisines', 'Price range', 'Aggregate rating']].copy()
    results['Recommendation Score'] = np.round(final_scores[top_indices], 3)
    return results.reset_index(drop=True)

# 5. Test & Output Results
print("--- Test Case 1: Italian Cuisine (Price Range 2) ---")
display(recommend_restaurants(user_cuisine="Italian", user_price_range=2, top_n=5))

print("\n--- Test Case 2: Chinese Cuisine (Price Range 3) ---")
display(recommend_restaurants(user_cuisine="Chinese", user_price_range=3, top_n=5))

--- Test Case 1: Italian Cuisine (Price Range 2) ---


,Restaurant Name,Cuisines,Price range,Aggregate rating,Recommendation Score
0,San Carlo,Italian,2,4.3,0.963
1,Trattoria Tiramisu,Italian,2,4.1,0.951
2,Trattoria Fresco,Italian,2,4.0,0.945
3,Sinyora's,Italian,2,4.0,0.945
4,Chilli Indiana,Italian,2,3.8,0.933



--- Test Case 2: Chinese Cuisine (Price Range 3) ---


,Restaurant Name,Cuisines,Price range,Aggregate rating,Recommendation Score
0,Din Tai Fung,Chinese,3,4.4,0.969
1,Mainland China,Chinese,3,4.3,0.963
2,Golden Dragon,Chinese,3,4.2,0.957
3,Yo! China,Chinese,3,4.1,0.951
4,The Cascade Restaurant,Chinese,3,4.1,0.951
